
# 13 — LangChain Prompt Engineering and LCEL

**Learning reference:**
[BrightPool — Prompt Engineering Course](https://github.com/BrightPool/udemy-prompt-engineering-course)

**Topics adapted from the course:**
- Chat models and message types
- Prompt templates and output parsing
- LangChain Expression Language (LCEL)
- Runnable sequences, parallel operations and branching
- Retrieval-Augmented Generation (RAG)
- Document processing, streaming and model configuration

## Goal

Build a small customer-support assistant using
LangChain and Gemini.

The application will:

1. Receive a customer-support question.
2. Retrieve relevant information from local policies.
3. Construct a reusable, evidence-grounded prompt.
4. Send that prompt to Gemini.
5. Parse the generated response.
6. Check the answer against expected behavior.

We will test two scenarios:

- A question supported by the available policies.
- A question that cannot be answered from those policies.

This notebook demonstrates actual prompt-to-model
interaction, not only deterministic Python workflows.

The retrieval component is intentionally simple and
does not claim to implement semantic vector search.



## 1. LangChain, LCEL and Prompt Engineering

**LangChain** provides reusable components for
building applications with language models.

**LCEL** (LangChain Expression Language) connects
components using the `|` operator.

Our workflow is:

Customer question
→ Retrieve policy
→ Format prompt
→ Gemini
→ Parse response

The prompt is responsible for defining the model's
behavior.

In this experiment, it must:

- Answer using only the retrieved policy.
- Avoid inventing information.
- Say when evidence is insufficient.
- Respond concisely.

The distinction is important:

**Python decides which documents to retrieve.**

**LangChain connects the processing steps.**

**Gemini generates the answer based on the prompt.**


In [7]:

%pip install -q "langchain-core>=1,<2" "langchain-openai>=1,<2"


In [1]:

from langchain_core.documents import Document

POLICIES = [
    Document(
        page_content=(
            "Customers may return unused products "
            "within 30 days of purchase. "
            "A receipt is required."
        ),
        metadata={
            "source": "returns_policy",
            "topic": "returns"
        }
    ),
    Document(
        page_content=(
            "Standard delivery usually takes "
            "3 to 5 business days."
        ),
        metadata={
            "source": "delivery_policy",
            "topic": "delivery"
        }
    ),
    Document(
        page_content=(
            "Customers who receive damaged products "
            "should contact support to discuss "
            "replacement options."
        ),
        metadata={
            "source": "damaged_items_policy",
            "topic": "damage"
        }
    )
]

for document in POLICIES:
    print(document.metadata["source"])
    print(document.page_content)
    print()

assert len(POLICIES) == 3

print("Three fictional policies loaded.")


returns_policy
Customers may return unused products within 30 days of purchase. A receipt is required.

delivery_policy
Standard delivery usually takes 3 to 5 business days.

damaged_items_policy
Customers who receive damaged products should contact support to discuss replacement options.

Three fictional policies loaded.



## 2. Retrieval Before Generation

Retrieval selects information relevant to a question.

In a complete RAG application, this might involve:

- Document loaders.
- Text splitting.
- Embeddings.
- Vector databases.
- Semantic similarity search.

For this small experiment, we use simple keyword
matching over three local policy documents.

The retriever returns an empty result when no
policy matches.

That is intentional: the model should not invent
an answer when the available information is
insufficient.

Retrieval prepares evidence. It does not itself
generate an answer.


In [3]:

def retrieve_documents(question):
    text = question.lower()

    if "return" in text or "refund" in text:
        topic = "returns"

    elif "damag" in text or "broken" in text:
        topic = "damage"

    elif (
        "deliver" in text
        or "shipping" in text
        or "parcel" in text
    ):
        topic = "delivery"

    else:
        return []

    return [
        document
        for document in POLICIES
        if document.metadata["topic"] == topic
    ]


def retrieve_context(question):
    documents = retrieve_documents(question)

    if not documents:
        return "NO RELEVANT POLICY FOUND"

    return "\n\n".join(
        f"Source: {doc.metadata['source']}\n"
        f"Content: {doc.page_content}"
        for doc in documents
    )


supported_question = (
    "Can I return an unused item after 20 days?"
)


unsupported_question = "Do you offer gift wrapping?"

print("Supported question context:")
print(retrieve_context(supported_question))

print("\nUnsupported question context:")
print(retrieve_context(unsupported_question))

assert len(retrieve_documents(supported_question)) == 1
assert retrieve_documents(unsupported_question) == []


Supported question context:
Source: returns_policy
Content: Customers may return unused products within 30 days of purchase. A receipt is required.

Unsupported question context:
NO RELEVANT POLICY FOUND



## 3. Design the Prompt

A reusable prompt separates instructions
from input data.

We will use `ChatPromptTemplate` with two roles.

**System message**

Defines the assistant's behavior and restrictions.

**Human message**

Contains the retrieved context and the
customer's actual question.

This design is useful because the same prompt
can process many different questions without
rewriting its instructions.

The prompt also includes an explicit fallback:

"I don't have enough policy information to answer that."

This allows us to test whether Gemini avoids
unsupported claims when evidence is missing.


In [4]:

from langchain_core.prompts import ChatPromptTemplate

FALLBACK = (
    "I don't have enough policy information "
    "to answer that."
)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a customer-support assistant.\n"
        "Use ONLY the supplied policy context.\n"
        "Do not invent prices, delivery options, "
        "refund decisions or order details.\n"
        "If the context does not directly support "
        "the answer, respond with exactly:\n"
        f"{FALLBACK}\n"
        "Otherwise, answer in one or two sentences."
    ),
    (
        "human",
        "Policy context:\n{context}\n\n"
        "Customer question:\n{question}"
    )
])

preview = prompt.invoke({
    "context": retrieve_context(supported_question),
    "question": supported_question
})

print(preview.to_string())
print("\nPrompt constructed without calling Gemini.")


System: You are a customer-support assistant.
Use ONLY the supplied policy context.
Do not invent prices, delivery options, refund decisions or order details.
If the context does not directly support the answer, respond with exactly:
I don't have enough policy information to answer that.
Otherwise, answer in one or two sentences.
Human: Policy context:
Source: returns_policy
Content: Customers may return unused products within 30 days of purchase. A receipt is required.

Customer question:
Can I return an unused item after 20 days?

Prompt constructed without calling Gemini.



## 4. Compose the LCEL Workflow

LangChain treats prompts, models and parsers
as reusable Runnable components.

Some useful Runnable interfaces are:

- `invoke()` — process one input.
- `batch()` — process multiple inputs.
- `stream()` — return output progressively,
  when supported.
- `with_config()` — apply execution settings.

We will compose the workflow using:

**RunnableParallel**

Produces both the retrieved context and
the original question.

**RunnableLambda**

Wraps a normal Python function as a Runnable.

**ChatPromptTemplate**

Formats the grounded instructions.

**ChatOpenAI**

Calls Gemini through its OpenAI-compatible API.

**StrOutputParser**

Converts the returned AI message into text.

The final pipeline is executed as a single chain.


In [8]:

from getpass import getpass

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

MODEL = "gemini-3.5-flash-lite"

gemini_key = getpass("Enter your Gemini API key: ")

gemini = ChatOpenAI(
    model=MODEL,
    api_key=gemini_key,
    base_url=(
        "https://generativelanguage.googleapis.com/"
        "v1beta/openai/"
    ),
    temperature=0,
    max_retries=0,
    timeout=120,
    use_responses_api=False
)

print("LangChain model configured:", MODEL)
print("No model request made yet.")


Enter your Gemini API key: ··········
LangChain model configured: gemini-3.5-flash-lite
No model request made yet.


In [9]:

from langchain_core.runnables import (
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough
)

prepare_inputs = RunnableParallel(
    context=RunnableLambda(retrieve_context),
    question=RunnablePassthrough()
)

rag_chain = (
    prepare_inputs
    | prompt
    | gemini
    | StrOutputParser()
)

print("LCEL pipeline created:")
print(
    "Question -> Retrieval -> Prompt -> "
    "Gemini -> Text parser"
)

print("\nNo Gemini request made yet.")


LCEL pipeline created:
Question -> Retrieval -> Prompt -> Gemini -> Text parser

No Gemini request made yet.


In [10]:

if "supported_answer" not in globals():
    supported_answer = rag_chain.invoke(
        supported_question
    )
    print("Gemini request completed.")
else:
    print("Using saved answer.")

print("\nQuestion:", supported_question)
print("Retrieved context:")
print(retrieve_context(supported_question))

print("\nGemini answer:")
print(supported_answer)

assert isinstance(supported_answer, str)
assert supported_answer.strip()


Gemini request completed.

Question: Can I return an unused item after 20 days?
Retrieved context:
Source: returns_policy
Content: Customers may return unused products within 30 days of purchase. A receipt is required.

Gemini answer:
Yes, you can return unused products within 30 days of purchase as long as you have a receipt.


In [11]:

if "unsupported_answer" not in globals():
    unsupported_answer = rag_chain.invoke(
        unsupported_question
    )
    print("Gemini request completed.")
else:
    print("Using saved answer.")

print("\nQuestion:", unsupported_question)
print("Retrieved context:")
print(retrieve_context(unsupported_question))

print("\nGemini answer:")
print(unsupported_answer)

assert isinstance(unsupported_answer, str)
assert unsupported_answer.strip()


Gemini request completed.

Question: Do you offer gift wrapping?
Retrieved context:
NO RELEVANT POLICY FOUND

Gemini answer:
I don't have enough policy information to answer that.



## 5. Evaluate the Generated Answers

We now evaluate the two model responses using
deterministic Python checks.

For the supported question, we check whether
the answer contains information about:

- The 30-day return period.
- The condition that the product is unused.
- The receipt requirement.

For the unsupported question, we check whether
Gemini returned the required fallback response.

These checks verify selected prompt requirements.

They are not a complete semantic evaluation and
cannot establish that every statement in the
generated answers is correct.

The evaluation does not make another model request.


In [13]:

supported_text = supported_answer.lower()

supported_checks = {
    "30-day period": (
        "30" in supported_text
        and "day" in supported_text
    ),
    "Unused condition": "unused" in supported_text,
    "Receipt requirement": "receipt" in supported_text
}

unsupported_check = (
    unsupported_answer.strip() == FALLBACK
)

print("SUPPORTED QUESTION")

for name, passed in supported_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

print("\nUNSUPPORTED QUESTION")
print(
    "Correct fallback:",
    "PASS" if unsupported_check else "FAIL"
)

passed_count = (
    sum(supported_checks.values())
    + int(unsupported_check)
)

print(f"\nChecks passed: {passed_count}/4")

if passed_count == 4:
    print("All defined response checks passed.")
else:
    print(
        "Some checks failed. Inspect the model "
        "responses before drawing conclusions."
    )

print("Additional Gemini requests: 0")


SUPPORTED QUESTION
30-day period: PASS
Unused condition: PASS
Receipt requirement: PASS

UNSUPPORTED QUESTION
Correct fallback: PASS

Checks passed: 4/4
All defined response checks passed.
Additional Gemini requests: 0



## Findings and Limitations

### What This Notebook Demonstrates

- Creating reusable prompts with `ChatPromptTemplate`.
- Separating system instructions from user input.
- Loading policy information into LangChain Documents.
- Retrieving context before model generation.
- Building LCEL pipelines with Runnable components.
- Combining context and user questions using
  `RunnableParallel`.
- Calling Gemini through LangChain's model interface.
- Parsing model-generated responses.
- Evaluating grounded answers against explicit rules.
- Testing the fallback behavior for missing evidence.

### Prompt Engineering Techniques

**Role instructions**

The system message defines the assistant's role
as a customer-support assistant.

**Grounding**

The prompt restricts responses to retrieved
policy information.

**Explicit constraints**

The prompt prohibits unsupported claims and
limits the response length.

**Abstention**

The model is instructed to acknowledge when
the context is insufficient.

**Evaluation**

Python checks whether selected prompt requirements
were satisfied.

### Additional LangChain Concepts

The course also explores several related topics:

- **Streaming:** Receiving model output incrementally.
- **Output parsing:** Converting model responses into
  structured application data.
- **Configurable fields:** Changing prompts or model
  settings at execution time.
- **Document loaders and text splitting:** Preparing
  longer documents for processing.
- **Document tagging:** Adding descriptive metadata.
- **Summarization chains:** Stuff, Map-Reduce and Refine.
- **Vector stores:** Searching documents using embeddings.
- **Index maintenance:** Detecting new, changed or
  removed documents.
- **Conversation history:** Retaining messages and
  contextualizing follow-up questions.

These concepts are introduced here but are not
all independently executed in this notebook.

Related techniques are demonstrated in earlier
repository notebooks or in the subsequent
LangGraph notebooks.

### Limitations

- The policies are fictional and locally defined.
- Retrieval uses keyword matching, not embeddings
  or semantic similarity.
- Only two customer questions are evaluated.
- The model is tested without repeated sampling.
- The checks are limited to specified keywords
  and exact fallback matching.
- The experiment does not measure retrieval recall
  or production-level answer accuracy.
- The improved reliability of grounding is not
  established through comparison with an
  ungrounded baseline.
  
### Observed Results

Two Gemini requests were executed through the
LangChain LCEL pipeline.

**Supported question:**
"Can I return an unused item after 20 days?"

Gemini answered that unused products can be returned
within 30 days with a receipt.

The response was consistent with the retrieved
returns policy and passed all three defined checks:
return period, unused condition and receipt requirement.

**Unsupported question:**
"Do you offer gift wrapping?"

No relevant policy was retrieved.

Gemini returned the exact required fallback:
"I don't have enough policy information to answer that."

**Evaluation summary:**
- Supported answer checks: 3/3 passed.
- Unsupported answer check: 1/1 passed.
- Total: 4/4 checks passed.

These results demonstrate successful grounding and
abstention on the two tested questions. They do not
establish general accuracy across other requests.


### Conclusion

This notebook demonstrates how LangChain and LCEL
can support a prompt-engineered AI application.

The workflow retrieves evidence, constructs a
grounded prompt, obtains a real Gemini response
and evaluates selected response requirements.

The central lesson is that prompt design,
retrieval and evaluation serve different roles:

- Retrieval supplies relevant information.
- Prompt engineering defines how the model should
  use that information.
- LCEL connects the processing steps.
- Evaluation checks whether the observed output
  meets the requirements.

The results describe this small experiment and
should not be generalized to broader applications.
